# 03 - Feature Engineering & Data Preparation: Pemerataan Faskes Indonesia

**Tujuan notebook**
1. Audit fitur dan missing value pada `master_provinsi_clean.csv`.
2. Menangani sisa missing value (RS Khusus, Balai Kesehatan) secara eksplisit.
3. Melakukan transformasi log pada variabel yang sangat miring.
4. Mendeteksi outlier dengan median/MAD (robust z-score), terutama DKI Jakarta.
5. Menstandarkan fitur (z-score) agar siap untuk *agglomerative clustering*.
6. Menyimpan dataset siap-klaster ke `data/processed/`.

**Alur**: 02 (cleaning) -> **03 (feature engineering)** -> 04 (EDA) -> 05 (clustering).

## 0. Setup

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 10

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
TABLE_DIR = ROOT / "outputs" / "tables"
FIG_DIR = ROOT / "outputs" / "figures"
for d in (TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT      :", ROOT)
print("PROCESSED :", PROCESSED)
print("TABLE_DIR :", TABLE_DIR)

## 1. Load & audit dataset

In [ ]:
df = pd.read_csv(PROCESSED / "master_provinsi_clean.csv")
print(f"Bentuk data: {df.shape[0]} provinsi x {df.shape[1]} kolom")
display(df.head())

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "n_unique": df.nunique(),
    "contoh": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
print("Kolom dengan missing value:")
display(audit[audit["n_missing"] > 0])

## 2. Pemilihan fitur: tiga lensa pemerataan

Clustering memakai fitur yang mewakili tiga dimensi berbeda agar tidak sekadar mengulang indikator yang sama:

| Lensa | Fitur | Alasan |
|---|---|---|
| Akses primer per kapita | `pkm_per100k` | Ketersediaan puskesmas relatif terhadap jumlah penduduk |
| Akses sekunder per kapita | `rs_per100k` | Ketersediaan rumah sakit rujukan relatif penduduk |
| Akses geografis | `faskes_per_1000km2` | Keterjangkauan fisik (kepadatan faskes per luas) |
| Cakupan desa (primer) | `desa_pkm_pct` | Persentase desa yang memiliki puskesmas |
| Cakupan desa (sekunder) | `desa_rs_pct` | Persentase desa yang memiliki rumah sakit |

`pkm_ratio_std` **tidak dipakai** karena proporsional sempurna dengan `pkm_per100k`
(keduanya mengukur puskesmas per penduduk), sehingga tidak menambah informasi.

## 3. Penanganan missing value

In [ ]:
# RS Khusus sudah diimputasi 0 pada notebook 02 dan diberi flag; cek konsistensinya.
print("Flag rsk_missing:")
print(df["rsk_missing"].value_counts(dropna=False).rename(index={False: "tidak missing", True: "missing -> diimputasi 0"}))

print("\nFlag desa_balai_missing:")
print(df["desa_balai_missing"].value_counts(dropna=False).rename(index={False: "tidak missing", True: "missing"}))

# Fitur clustering tidak memiliki missing; kolom balai tidak dipakai dalam clustering.
print("\nMissing pada fitur clustering:",
      df[["pkm_per100k", "rs_per100k", "faskes_per_1000km2", "desa_pkm_pct", "desa_rs_pct"]].isna().sum().to_dict())

**Catatan**: `desa_balai` dibiarkan missing dan tidak dipakai karena bukan bagian dari fitur
pemerataan utama. Untuk analisis lanjutan, missing ini dapat diperlakukan sebagai *missing
not at random* (balai kesehatan memang tidak ada / tidak tercatat).

## 4. Transformasi log untuk variabel miring

In [ ]:
BASE_COLS = ["pkm_per100k", "rs_per100k", "faskes_per_1000km2", "desa_pkm_pct", "desa_rs_pct"]

skew_table = pd.DataFrame({
    "skewness": df[BASE_COLS].skew(),
    "min": df[BASE_COLS].min(),
    "max": df[BASE_COLS].max(),
})
display(skew_table.round(2))

# faskes_per_1000km2 sangat miring (DKI Jakarta ekstrem) -> log1p
df["log_faskes_per_1000km2"] = np.log1p(df["faskes_per_1000km2"])

CLUSTER_COLS = ["pkm_per100k", "rs_per100k", "log_faskes_per_1000km2", "desa_pkm_pct", "desa_rs_pct"]
print("Fitur final untuk clustering:", CLUSTER_COLS)

Log1p dipilih karena aman untuk nilai 0 dan meredam pengaruh ekstrem tanpa membuang
observasi. Untuk fitur lain, skew masih wajar (< 1.1) sehingga tetap memakai skala asli.

## 5. Deteksi outlier (robust z-score / MAD)

In [ ]:
med = df[BASE_COLS].median()
mad = (df[BASE_COLS] - med).abs().median()
robust_z = (df[BASE_COLS] - med) / mad.replace(0, np.nan) * 1.4826

outlier_mask = (robust_z.abs() > 3.5).any(axis=1)
df["is_outlier"] = outlier_mask

print("Provinsi terdeteksi outlier (|robust z| > 3.5):")
display(df.loc[outlier_mask, ["Provinsi"] + BASE_COLS].round(2))
print(f"Jumlah provinsi outlier: {int(outlier_mask.sum())}")

print("\nJumlah provinsi ekstrem per fitur:")
display((robust_z.abs() > 3.5).sum().rename("n_provinsi_ekstrem"))

print("\nRobust z-score per provinsi (5 nilai terbesar):")
display(robust_z.abs().max(axis=1).sort_values(ascending=False).head(5).round(2))

**Interpretasi**: Data memang *heavy-tailed* — beberapa provinsi ekstrem pada fitur
berbeda. DKI Jakarta paling ekstrem pada `faskes_per_1000km2` (kepadatan faskes sangat
tinggi) dan `desa_rs_pct`, tetapi justru sangat rendah pada `pkm_per100k` karena struktur
layanan kota yang bertumpu pada RS dan klinik. Provinsi kepulauan/pedalaman juga ekstrem
pada rasio per kapita.

Outlier **tidak dibuang**; pada notebook 05 observasi dibiarkan membentuk tipe tersendiri
agar tidak mendistorsi segmentasi provinsi lain. Flag `is_outlier` disimpan untuk
keperluan uji sensitivitas.

## 6. Standardisasi z-score

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[CLUSTER_COLS])

SCALED_COLS = [
    "scaled_pkm_per100k",
    "scaled_rs_per100k",
    "scaled_log_density",
    "scaled_desa_pkm_pct",
    "scaled_desa_rs_pct",
]
scaled_df = pd.DataFrame(X_scaled, columns=SCALED_COLS, index=df.index)
df = pd.concat([df, scaled_df], axis=1)
display(df[["Provinsi"] + SCALED_COLS].head())
print("Rata-rata setelah scaling:", np.round(X_scaled.mean(axis=0), 6))
print("Std setelah scaling     :", np.round(X_scaled.std(axis=0), 6))

## 7. Korelasi antarfitur

In [ ]:
corr = df[BASE_COLS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 5.5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            center=0, square=True, mask=mask, cbar_kws={"shrink": .6}, ax=ax)
ax.set_title("Korelasi Spearman — Fitur Pemerataan", weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "korelasi_fitur_clustering.png", dpi=150, bbox_inches="tight")
plt.show()

print("Pasangan fitur dengan |korelasi| > 0.7:")
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f"  {corr.columns[i]} <-> {corr.columns[j]}: {corr.iloc[i, j]:.2f}")

Korelasi tinggi tidak dihindari sepenuhnya karena tiap fitur mewakili lensa berbeda
(per kapita vs cakupan desa). Namun tidak ada pasangan yang berkorelasi sempurna (>0.95),
sehingga kelima fitur tetap layak dipakai bersama dalam Ward clustering.

## 8. Simpan dataset siap analisis

In [ ]:
# Dataset lengkap (raw + log + flag + scaled)
features_path = PROCESSED / "faskes_features.csv"
df.to_csv(features_path, index=False, encoding="utf-8")

# Matriks khusus clustering
cluster_matrix = df[["Provinsi"] + CLUSTER_COLS + SCALED_COLS + ["is_outlier"]].copy()
matrix_path = PROCESSED / "faskes_clustering_scaled.csv"
cluster_matrix.to_csv(matrix_path, index=False, encoding="utf-8")

# Tabel deskriptif
desc = df[BASE_COLS].describe().T
desc["skewness"] = df[BASE_COLS].skew()
desc_path = TABLE_DIR / "deskriptif_fitur.csv"
desc.to_csv(desc_path, encoding="utf-8")

print("Tersimpan:")
for p in (features_path, matrix_path, desc_path):
    print(" -", p)
display(desc.round(3))

## 9. Ringkasan

- 38 provinsi, tanpa missing pada fitur clustering.
- `faskes_per_1000km2` di-log karena sangat miring; DKI Jakarta tetap outlier ekstrem.
- Lima fitur final mencakup akses primer, sekunder, geografis, dan cakupan desa.
- Output: `data/processed/faskes_features.csv` dan `data/processed/faskes_clustering_scaled.csv`.

**Next**: notebook 04 memvisualisasikan sebaran dan pola regional; notebook 05 menjalankan
agglomerative clustering dan analisis ketimpangan.